In [0]:
%run ../functions/functions

In [0]:
# Nome do banco de dados onde a tabela será criada ou utilizada
database_name = "fato"

# Nome da tabela de exportações por município
table_name = "ft_exportacoes_mun"

# Caminho alvo no formato database.tabela para salvar ou acessar a tabela
target_path = f"{database_name}.{table_name}"

# Nome da chave primária da tabela
pk = "SK_EXPORTACAO"

In [0]:
# Bases utilizadas no relacionamento exportacao por municipio
# Caminho do arquivo Delta referente à exportação por município consolidada na camada silver
silver_path_s = f"abfss://silver@stgbbb.dfs.core.windows.net/balancacomercial/EXP_MUN_CONSOLIDADA/"

In [0]:
# Lê os dados de exportação por município a partir do caminho Delta no Silver Layer
df_exp_mun = spark.read.format("delta").load(silver_path_s)

In [0]:
# Cria ou substitui uma view temporária chamada "df_exp_mun" a partir do DataFrame df_exp_mun
df_exp_mun.createOrReplaceTempView("df_exp_mun")

In [0]:
# Consulta SQL para selecionar e transformar dados da tabela temporária df_exp_mun
query = """
select
    e.SH4,  -- Código SH4 do produto exportado
    concat(e.CO_ANO,'-',e.CO_MES) as ano_mes,  -- Ano e mês concatenados no formato 'YYYY-MM'
    e.VL_FOB,  -- Valor FOB da exportação

    -- Relacionamento com tabelas geográficas:
    e.CO_PAIS,  -- Código do país de destino
    e.CO_MUN,   -- Código do município de origem

    e.KG_LIQUIDO  -- Peso líquido da exportação

from df_exp_mun e  -- Fonte dos dados: tabela temporária df_exp_mun
"""

In [0]:
# Executa a consulta SQL definida na variável 'query' e armazena o resultado no DataFrame 'df_join'
df_join = spark.sql(query)

In [0]:
df_join.count()

In [0]:
save_hive_table(df_join, target_path, pk)